# HiddenExtension2 결과 분석

분석 구성:
1. 전체 실험 결과 요약
2. 모델 선택 & 로드 (아래 **Configuration** 셀에서 설정)
3. 통계 검정 (P-value)
4. 변수 중요도 (θ 가중치 + 퍼미테이션)
5. 잔차 분석

In [ ]:
import os, sys, json, glob
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy.stats as stats
from torch.utils.data import DataLoader

# ── 경로 설정 (config.py 위치로 HE2_DIR 확정) ─────────────────────────────
_cwd = os.getcwd()
if os.path.isfile(os.path.join(_cwd, 'config.py')):
    HE2_DIR = _cwd
elif os.path.isfile(os.path.join(_cwd, 'HiddenExtension2', 'config.py')):
    HE2_DIR = os.path.join(_cwd, 'HiddenExtension2')
else:
    HE2_DIR = '/workspace/ST-GNN Modeling/HiddenExtension2'

ROOT_DIR = os.path.abspath(os.path.join(HE2_DIR, '..'))

# 이전 실행에서 캐시된 잘못된 모듈 제거 후 HE2_DIR 최우선 삽입
for _mod in ['model', 'config', 'he_dataset', 'utils']:
    sys.modules.pop(_mod, None)
for _p in [ROOT_DIR, HE2_DIR]:
    if _p in sys.path:
        sys.path.remove(_p)
sys.path.insert(0, ROOT_DIR)
sys.path.insert(0, HE2_DIR)

from config import H_DIM, ATT_HIDDEN, DROPOUT
from he_dataset import PseudoGridDataset, feature_names_for_mode
from model import JointHiddenExtensionModel

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CKPT_ROOT = os.path.join(HE2_DIR, 'checkpoints')

print(f'HE2_DIR  : {HE2_DIR}')
print(f'Device   : {DEVICE}')
print('✓ JointHiddenExtensionModel 임포트 성공')

## ⚙️ Configuration — 분석할 모델을 여기서 선택하세요

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  방법 1) 실험 이름 직접 입력
#   SEL_EXP = "xnone_attnfull_lurlinear_r8_lam0.3"
#
#  방법 2) 아래 목록에서 Rank 번호 입력 (1~N)
#   SEL_EXP = 1
#
#  방법 3) direct_mae 기준 자동 최고
#   SEL_EXP = "best"
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

SEL_EXP = 'best'   # ← 여기서 변경

# ── 사용 가능한 실험 목록 출력 ──────────────────────────────────────────
_files = sorted(glob.glob(os.path.join(CKPT_ROOT, '**/metrics.json'), recursive=True))
_records = []
for _f in _files:
    with open(_f) as fp:
        _d = json.load(fp)
    _d['experiment'] = _f.split('checkpoints/')[1].replace('/metrics.json', '')
    _records.append(_d)
_df = pd.DataFrame(_records).sort_values('direct_mae').reset_index(drop=True)
_df.index += 1

print('── 사용 가능한 실험 목록 (direct_mae 순) ──────────────────────────────')
print(f'{"#":<5} {"실험 이름":<55} {"direct_MAE":<12} {"cross_MAE"}')
print('-' * 88)
for _i, _row in _df.iterrows():
    print(f'{_i:<5} {_row["experiment"]:<55} {_row["direct_mae"]:.4f}       {_row["cross_mae"]:.4f}')

# ── SEL_EXP 해석 ─────────────────────────────────────────────────────────
if SEL_EXP == 'best':
    _sel_row = _df.iloc[0]
elif isinstance(SEL_EXP, int):
    assert 1 <= SEL_EXP <= len(_df), f'Rank는 1~{len(_df)} 사이여야 합니다'
    _sel_row = _df.loc[SEL_EXP]
else:
    _match = _df[_df['experiment'] == SEL_EXP]
    assert len(_match) > 0, f"'{SEL_EXP}' 실험을 찾을 수 없습니다"
    _sel_row = _match.iloc[0]

SEL_DIR = _sel_row['experiment']
SEL_CFG = {
    'x_mode':    _sel_row.get('x_mode', 'none'),
    'attn_mode': _sel_row.get('attn_mode', 'full'),
    'lur_mode':  _sel_row.get('lur_mode', 'linear'),
    'r_dim':     int(_sel_row.get('r_dim', 8)),
    'lam':       float(_sel_row.get('lam', 0.3)),
}

print(f"\n{'━'*60}")
print(f"  선택된 실험 : {SEL_DIR}")
print(f"  x_mode     : {SEL_CFG['x_mode']}")
print(f"  attn_mode  : {SEL_CFG['attn_mode']}")
print(f"  lur_mode   : {SEL_CFG['lur_mode']}")
print(f"  r_dim      : {SEL_CFG['r_dim']}")
print(f"  lam        : {SEL_CFG['lam']}")
print(f"  direct_MAE : {_sel_row['direct_mae']:.4f}")
print(f"  cross_MAE  : {_sel_row['cross_mae']:.4f}")
print(f"{'━'*60}")

## 1. 전체 실험 결과 요약

In [ ]:
files = glob.glob(os.path.join(CKPT_ROOT, '**/metrics.json'), recursive=True)
records = []
for f in files:
    with open(f) as fp:
        d = json.load(fp)
    d['experiment'] = f.split('checkpoints/')[1].replace('/metrics.json', '')
    records.append(d)

df_all = pd.DataFrame(records).sort_values('direct_mae').reset_index(drop=True)
df_all.index += 1

display_cols = ['experiment', 'x_mode', 'attn_mode', 'lur_mode', 'r_dim', 'lam',
                'direct_mae', 'direct_rmse', 'direct_r2', 'cross_mae', 'cross_rmse', 'cross_r2', 'stgnn_mae']
df_show = df_all[[c for c in display_cols if c in df_all.columns]].copy()

def highlight_sel(row):
    color = 'background-color: #d4edda' if row['experiment'] == SEL_DIR else ''
    return [color] * len(row)

df_show.style \
    .apply(highlight_sel, axis=1) \
    .format({'direct_mae': '{:.4f}', 'direct_rmse': '{:.4f}', 'direct_r2': '{:.4f}',
             'cross_mae': '{:.4f}', 'cross_rmse': '{:.4f}', 'cross_r2': '{:.4f}',
             'stgnn_mae': '{:.4f}', 'lam': '{:.1f}'})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = df_all['experiment'].apply(lambda x: '#e74c3c' if x == SEL_DIR else '#3498db')
ax.scatter(df_all['direct_mae'], df_all['cross_mae'], c=colors, s=80, zorder=3)

baseline = df_all['stgnn_mae'].iloc[0]
ax.axvline(baseline, color='gray', linestyle='--', linewidth=1, label=f'ST-GNN baseline MAE={baseline:.4f}')

for _, row in df_all.iterrows():
    label = row['experiment'].replace('attnfull_', '').replace('lurlinear_', '')
    ax.annotate(label, (row['direct_mae'], row['cross_mae']),
                fontsize=6, xytext=(3, 3), textcoords='offset points')

ax.scatter([], [], c='#e74c3c', s=80, label=f'선택: {SEL_DIR}')
ax.scatter([], [], c='#3498db', s=80, label='Others')
ax.set_xlabel('Direct MAE', fontsize=11)
ax.set_ylabel('Cross MAE', fontsize=11)
ax.set_title('HiddenExtension2 — Direct vs Cross MAE (전체 실험)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. 선택된 모델 로드 및 테스트셋 추론

In [ ]:
def collate_fn(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}

train_ds = PseudoGridDataset('train', x_mode=SEL_CFG['x_mode'])
test_ds  = PseudoGridDataset('test',  x_mode=SEL_CFG['x_mode'],
                              X_scaler=train_ds.X_scaler,
                              wind_scaler=train_ds.wind_scaler)
test_loader = DataLoader(test_ds, batch_size=1024, shuffle=False,
                         num_workers=0, collate_fn=collate_fn)

x_dim     = train_ds.x_dim
ckpt_path = os.path.join(CKPT_ROOT, SEL_DIR, 'best_model.pt')

model = JointHiddenExtensionModel(
    h_dim=H_DIM, x_dim=x_dim,
    r_dim=SEL_CFG['r_dim'],
    att_hidden=ATT_HIDDEN, dropout=DROPOUT,
    lur_mode=SEL_CFG['lur_mode'],
    attn_mode=SEL_CFG['attn_mode'],
).to(DEVICE)
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
model.eval()

feat_names = feature_names_for_mode(SEL_CFG['x_mode'])
print(f'모델 로드 완료 : {ckpt_path}')
print(f'파라미터 수    : {sum(p.numel() for p in model.parameters()):,}')
print(f'x_dim          : {x_dim}  →  피처: {feat_names if feat_names else "없음"}')
print(f'테스트 샘플 수 : {len(test_ds):,}')

In [ ]:
# 추론 + r_direct 벡터 수집 (변수 중요도 분석용)
r_direct_list = []

def _hook_r_direct(module, input, output):
    r_direct_list.append(output.detach().cpu())

hook = model.compressor_direct.register_forward_hook(_hook_r_direct)

d_preds, c_preds, trues = [], [], []
with torch.no_grad():
    for batch in test_loader:
        pd_, pc = model(
            batch['h_sources'].to(DEVICE), batch['coords_target'].to(DEVICE),
            batch['coords_sources'].to(DEVICE), batch['X_target'].to(DEVICE),
            batch['X_sources'].to(DEVICE), batch['wind_sources'].to(DEVICE),
            batch['h_target'].to(DEVICE),
        )
        d_preds.append(pd_.cpu().numpy())
        c_preds.append(pc.cpu().numpy())
        trues.append(batch['pm_target'].numpy())

hook.remove()

y_pred    = np.concatenate(d_preds)
y_cross   = np.concatenate(c_preds)
y_true    = np.concatenate(trues)
R_direct  = torch.cat(r_direct_list)
residuals = y_pred - y_true

mae   = float(np.mean(np.abs(residuals)))
rmse  = float(np.sqrt(np.mean(residuals**2)))
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y_true - y_true.mean())**2)
r2    = float(1 - ss_res / ss_tot)

print(f'\n=== 테스트셋 성능 ===')
print(f'  Direct  MAE : {mae:.4f}')
print(f'  Direct  RMSE: {rmse:.4f}')
print(f'  Direct  R²  : {r2:.4f}')
print(f'  샘플 수     : {len(y_true):,}')

## 3. 통계 검정 (P-value)

| 검정 | 귀무가설 (H₀) |
|------|----------------|
| Pearson 상관 | 예측값과 실제값 간 선형 상관 없음 |
| Spearman 상관 | 예측값과 실제값 간 단조 상관 없음 |
| t-검정 (잔차) | 잔차 평균 = 0 (편향 없음) |
| Wilcoxon 검정 | 잔차 중앙값 = 0 |
| KS 정규성 | 잔차가 정규분포를 따름 |
| Permutation test | 모델이 랜덤 예측과 유의하게 다름 |

In [ ]:
r_pearson,  p_pearson  = stats.pearsonr(y_true, y_pred)
r_spearman, p_spearman = stats.spearmanr(y_true, y_pred)
t_stat,     p_ttest    = stats.ttest_1samp(residuals, 0)
w_stat,     p_wilcoxon = stats.wilcoxon(residuals, alternative='two-sided')

N = len(residuals)
if N <= 5000:
    norm_stat, p_norm = stats.shapiro(residuals)
    norm_test_name = 'Shapiro-Wilk'
else:
    res_z = (residuals - residuals.mean()) / residuals.std()
    norm_stat, p_norm = stats.kstest(res_z, 'norm')
    norm_test_name = 'Kolmogorov-Smirnov'

np.random.seed(42)
N_PERM = 1000
perm_maes = np.array([np.mean(np.abs(np.random.permutation(y_pred) - y_true))
                      for _ in range(N_PERM)])
p_perm = float(np.mean(perm_maes <= mae))

sig = lambda p: '**' if p < 0.01 else ('*' if p < 0.05 else 'ns')
results_stat = [
    ('Pearson r',     f'{r_pearson:.4f}',  f'{p_pearson:.2e}',  sig(p_pearson)),
    ('Spearman ρ',    f'{r_spearman:.4f}', f'{p_spearman:.2e}', sig(p_spearman)),
    ('t-test (잔차)', f'{t_stat:.4f}',     f'{p_ttest:.2e}',    sig(p_ttest)),
    ('Wilcoxon',      f'{w_stat:.1f}',     f'{p_wilcoxon:.2e}', sig(p_wilcoxon)),
    (norm_test_name,  f'{norm_stat:.4f}',  f'{p_norm:.2e}',     '정규' if p_norm > 0.05 else '비정규'),
    ('Permutation',   f'MAE={mae:.4f}',    f'{p_perm:.4f}',     sig(p_perm)),
]
df_stat = pd.DataFrame(results_stat, columns=['검정', '통계량', 'p-value', '결론'])
print(df_stat.to_string(index=False))
print('\n* p<0.05  ** p<0.01  ns: not significant')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(perm_maes, bins=40, color='#95a5a6', edgecolor='white', label='Permuted MAE 분포')
ax.axvline(mae, color='#e74c3c', linewidth=2.5, label=f'Actual MAE = {mae:.4f}')
ax.axvline(np.percentile(perm_maes, 5), color='#e67e22', linewidth=1.5,
           linestyle='--', label=f'5th pct = {np.percentile(perm_maes,5):.4f}')
ax.set_xlabel('MAE', fontsize=11)
ax.set_ylabel('빈도', fontsize=11)
ax.set_title(f'Permutation Test (n={N_PERM})  —  empirical p = {p_perm:.4f}', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. 변수 중요도 (Variable Importance)

- **x_mode=`none`**: r_dim 잠재 차원의 θ 가중치 + 마스킹 퍼미테이션 중요도
- **x_mode=그 외**: LUR 피처 β 가중치 + r_dim θ 가중치 + 퍼미테이션 중요도

In [ ]:
r_dim      = SEL_CFG['r_dim']
dim_labels = [f'r{i}' for i in range(r_dim)]

if SEL_CFG['lur_mode'] == 'linear':
    theta_w = model.theta.weight.squeeze().detach().cpu().numpy()
    theta_b = float(model.theta.bias.item())
    rel = np.abs(theta_w) / (np.sum(np.abs(theta_w)) + 1e-12)
    print(f'θ bias : {theta_b:.4f}')
    print(f'\nr_dim θ 가중치 기여 비율:')
    for lbl, w, r in zip(dim_labels, theta_w, rel):
        print(f'  {lbl}: {w:+.4f}  ({r:.1%})  {"█" * int(r * 40)}')

    if x_dim > 0 and hasattr(model, 'beta'):
        beta_w = model.beta.weight.squeeze().detach().cpu().numpy()
        rel_b  = np.abs(beta_w) / (np.sum(np.abs(beta_w)) + 1e-12)
        print(f'\nβ (LUR 피처) 가중치:')
        for fn, bw, rb in zip(feat_names, beta_w, rel_b):
            print(f'  {fn:<20}: {bw:+.4f}  ({rb:.1%})  {"█" * int(rb * 40)}')
else:
    print('lur_mode=mlp → 선형 계수 없음 (퍼미테이션 중요도로만 분석)')

In [ ]:
theta_tensor = model.theta.weight.detach().cpu()
theta_bias   = model.theta.bias.detach().cpu()

# r_dim 퍼미테이션 중요도
r_perm_importance = []
for i in range(r_dim):
    R_masked = R_direct.clone()
    R_masked[:, i] = 0.0
    y_masked = (R_masked @ theta_tensor.T + theta_bias).squeeze().numpy()
    if x_dim > 0 and hasattr(model, 'beta'):
        r_orig = (R_direct @ theta_tensor.T + theta_bias).squeeze().numpy()
        y_masked = y_masked + (y_pred - r_orig)
    r_perm_importance.append(float(np.mean(np.abs(y_masked - y_true))) - mae)
r_perm_importance = np.array(r_perm_importance)

# β 피처 퍼미테이션 중요도 (x_mode≠none, linear)
feat_perm_importance = np.array([])
if x_dim > 0 and hasattr(model, 'beta') and SEL_CFG['lur_mode'] == 'linear':
    beta_tensor  = model.beta.weight.detach().cpu()
    X_test_all   = torch.stack([test_ds[j]['X_target'] for j in range(len(test_ds))])
    r_pred_base  = (R_direct @ theta_tensor.T + theta_bias).squeeze().numpy()
    feat_perm_importance = []
    for i in range(x_dim):
        X_masked = X_test_all.clone()
        X_masked[:, i] = 0.0
        y_fi = r_pred_base + (X_masked @ beta_tensor.T).squeeze().numpy()
        feat_perm_importance.append(float(np.mean(np.abs(y_fi - y_true))) - mae)
    feat_perm_importance = np.array(feat_perm_importance)

print('r_dim 퍼미테이션 중요도 (차원 masking 시 ΔMAE):')
for lbl, d in zip(dim_labels, r_perm_importance):
    print(f'  {lbl}: ΔMAE={d:+.4f}  {"█" * int(abs(d)/(max(np.abs(r_perm_importance))+1e-12)*30)}')

if len(feat_perm_importance) > 0:
    print('\nLUR 피처 퍼미테이션 중요도 (피처 masking 시 ΔMAE):')
    for fn, d in zip(feat_names, feat_perm_importance):
        print(f'  {fn:<20}: ΔMAE={d:+.4f}  {"█" * int(abs(d)/(max(np.abs(feat_perm_importance))+1e-12)*30)}')

In [ ]:
has_beta = x_dim > 0 and hasattr(model, 'beta') and SEL_CFG['lur_mode'] == 'linear'
n_plots  = 2 + (1 if has_beta else 0)
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 4))
if n_plots == 1:
    axes = [axes]

if SEL_CFG['lur_mode'] == 'linear':
    ax = axes[0]
    colors_w = ['#e74c3c' if v > 0 else '#3498db' for v in theta_w]
    bars = ax.bar(dim_labels, theta_w, color=colors_w, edgecolor='white', width=0.6)
    ax.axhline(0, color='black', linewidth=0.8)
    for bar, val in zip(bars, theta_w):
        ax.text(bar.get_x() + bar.get_width()/2, val + (0.002 if val>=0 else -0.002),
                f'{val:.3f}', ha='center', va='bottom' if val>=0 else 'top', fontsize=8)
    ax.set_title('θ 가중치 (r_dim 선형 계수)', fontsize=11)
    ax.set_ylabel('θ weight', fontsize=10)
    ax.set_xlabel('r_dim 차원', fontsize=10)
    ax.grid(axis='y', alpha=0.3)

ax = axes[1]
colors_p = ['#e74c3c' if v > 0 else '#2ecc71' for v in r_perm_importance]
bars = ax.bar(dim_labels, r_perm_importance, color=colors_p, edgecolor='white', width=0.6)
ax.axhline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, r_perm_importance):
    ax.text(bar.get_x() + bar.get_width()/2, val + (0.0002 if val>=0 else -0.0002),
            f'{val:+.4f}', ha='center', va='bottom' if val>=0 else 'top', fontsize=8)
ax.set_title('r_dim 퍼미테이션 중요도 (ΔMAE)', fontsize=11)
ax.set_ylabel('ΔMAE (↑ = 중요)', fontsize=10)
ax.set_xlabel('r_dim 차원', fontsize=10)
ax.grid(axis='y', alpha=0.3)

if has_beta:
    ax = axes[2]
    colors_f = ['#e74c3c' if v > 0 else '#2ecc71' for v in feat_perm_importance]
    bars = ax.bar(feat_names, feat_perm_importance, color=colors_f, edgecolor='white', width=0.6)
    ax.axhline(0, color='black', linewidth=0.8)
    for bar, val in zip(bars, feat_perm_importance):
        ax.text(bar.get_x() + bar.get_width()/2, val + (0.0002 if val>=0 else -0.0002),
                f'{val:+.4f}', ha='center', va='bottom' if val>=0 else 'top', fontsize=8)
    ax.set_title('LUR 피처 퍼미테이션 중요도 (ΔMAE)', fontsize=11)
    ax.set_ylabel('ΔMAE (↑ = 중요)', fontsize=10)
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'변수 중요도 — {SEL_DIR}', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# x_mode ablation 비교
x_mode_order  = ['none', 'satellite', 'no_building', 'all']
x_mode_labels = {'none': '없음', 'satellite': 'NDVI+IBI', 'no_building': 'NDVI+IBI+LC', 'all': '전체(9차원)'}

ablation_data = []
for _, row in df_all.iterrows():
    if (row.get('lur_mode') == 'linear' and row.get('attn_mode') == 'full'
            and row.get('r_dim') == 8 and row.get('lam') == 0.3
            and row.get('x_mode') in x_mode_order):
        ablation_data.append(row)

if ablation_data:
    df_ab = pd.DataFrame(ablation_data).set_index('x_mode').reindex(x_mode_order)
    stgnn_base = df_ab['stgnn_mae'].iloc[0]
    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(len(x_mode_order))
    bars_d = ax.bar(x - 0.2, df_ab['direct_mae'], 0.35, label='Direct MAE', color='#3498db', alpha=0.85)
    bars_c = ax.bar(x + 0.2, df_ab['cross_mae'],  0.35, label='Cross MAE',  color='#e67e22', alpha=0.85)
    ax.axhline(stgnn_base, color='gray', linestyle='--', linewidth=1.5,
               label=f'ST-GNN baseline ({stgnn_base:.4f})')
    ax.set_xticks(x)
    ax.set_xticklabels([x_mode_labels.get(m, m) for m in x_mode_order], fontsize=10)
    ax.set_ylabel('MAE', fontsize=11)
    ax.set_title('LUR 피처 종류별 성능 비교 (r=8, lam=0.3, attn=full)', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(2.4, 4.0)
    for bar in list(bars_d) + list(bars_c):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('x_mode ablation 데이터 없음')

## 5. 잔차 분석 (Residual Analysis)

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, :])
vmin, vmax = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
h2d = ax1.hist2d(y_true, y_pred, bins=80, cmap='Blues', range=[[vmin,vmax],[vmin,vmax]])
plt.colorbar(h2d[3], ax=ax1, label='Count')
ax1.plot([vmin,vmax], [vmin,vmax], 'r--', linewidth=1.5, label='Perfect prediction')
ax1.set_xlabel('실측 PM10', fontsize=11)
ax1.set_ylabel('예측 PM10', fontsize=11)
ax1.set_title(f'예측 vs 실측  (MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f})', fontsize=12)
ax1.legend(fontsize=9)

ax2 = fig.add_subplot(gs[1, 0])
ax2.hist(residuals, bins=60, color='#3498db', edgecolor='white', alpha=0.85, density=True)
xr = np.linspace(residuals.min(), residuals.max(), 300)
ax2.plot(xr, stats.norm.pdf(xr, residuals.mean(), residuals.std()), 'r-', linewidth=2, label='Normal fit')
ax2.axvline(0, color='black', linewidth=1)
ax2.set_xlabel('잔차 (예측 - 실측)', fontsize=10)
ax2.set_ylabel('밀도', fontsize=10)
ax2.set_title(f'잔차 히스토그램\nmean={residuals.mean():.4f}, std={residuals.std():.4f}', fontsize=10)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

ax3 = fig.add_subplot(gs[1, 1])
res_sorted = np.sort(residuals)
n = len(res_sorted)
theoretical_q = stats.norm.ppf((np.arange(1, n+1) - 0.5) / n)
ax3.scatter(theoretical_q, res_sorted, s=1, alpha=0.3, color='#3498db')
q1, q3   = np.percentile(residuals, [25, 75])
tq1, tq3 = stats.norm.ppf([0.25, 0.75])
slope = (q3 - q1) / (tq3 - tq1)
xl = np.array([theoretical_q.min(), theoretical_q.max()])
ax3.plot(xl, slope * xl + (q1 - slope * tq1), 'r-', linewidth=2)
ax3.set_xlabel('이론적 분위수 (Normal)', fontsize=10)
ax3.set_ylabel('실제 잔차 분위수', fontsize=10)
ax3.set_title('Q-Q 플롯', fontsize=10)
ax3.grid(alpha=0.3)

ax4 = fig.add_subplot(gs[1, 2])
ax4.scatter(y_pred, residuals, s=2, alpha=0.15, color='#2c3e50')
ax4.axhline(0, color='red', linewidth=1.5)
ax4.axhline(residuals.mean() + 2*residuals.std(), color='orange', linewidth=1, linestyle='--')
ax4.axhline(residuals.mean() - 2*residuals.std(), color='orange', linewidth=1, linestyle='--', label='±2σ')
ax4.set_xlabel('예측값', fontsize=10)
ax4.set_ylabel('잔차', fontsize=10)
ax4.set_title('잔차 vs 예측값', fontsize=10)
ax4.legend(fontsize=8)
ax4.grid(alpha=0.3)

fig.suptitle(f'잔차 분석 — {SEL_DIR}', fontsize=13, y=1.01)
plt.show()

In [ ]:
bins    = [0, 15, 30, 50, 75, 100, 150, 1000]
labels  = ['0-15', '15-30', '30-50', '50-75', '75-100', '100-150', '150+']
bin_idx = np.digitize(y_true, bins) - 1

bin_stats = []
for i, lbl in enumerate(labels):
    mask = bin_idx == i
    if mask.sum() == 0:
        continue
    bin_stats.append({'구간': lbl, '샘플수': int(mask.sum()),
                      'MAE': float(np.mean(np.abs(residuals[mask]))),
                      '평균잔차(bias)': float(np.mean(residuals[mask]))})

df_bin = pd.DataFrame(bin_stats)
print('PM10 농도 구간별 MAE:')
display(df_bin.style.format({'MAE': '{:.4f}', '평균잔차(bias)': '{:.4f}'}))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(df_bin))
ax.bar(x, df_bin['MAE'], color='#3498db', alpha=0.8, label='MAE')
ax2b = ax.twinx()
ax2b.bar(x, df_bin['평균잔차(bias)'], color='#e74c3c', alpha=0.5, label='Bias')
ax2b.axhline(0, color='red', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(df_bin['구간'], fontsize=10)
ax.set_ylabel('MAE', color='#3498db', fontsize=10)
ax2b.set_ylabel('평균 잔차 (Bias)', color='#e74c3c', fontsize=10)
ax.set_title('PM10 농도 구간별 MAE & Bias', fontsize=12)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=9, loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()